# 04 — SQL Analytics with DuckDB

This notebook validates and explores the Gold Star Schema:

- `dim_date`
- `dim_vehicle`
- `fct_fipe_prices`

Power BI should consume these three tables directly.


In [ ]:
from pathlib import Path

from fipe_pipeline.analytics_views import create_analytics_views
from fipe_pipeline.duckdb_layer import (
    connect_duckdb,
    register_parquet_views,
    validate_duckdb_layer,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [ ]:
con = connect_duckdb()
register_parquet_views(con)
create_analytics_views(con)

validate_duckdb_layer(con)

## 1. Gold Star Schema objects


In [ ]:
con.sql("""
SELECT
    table_name,
    table_type
FROM information_schema.tables
WHERE table_name IN (
    'dim_date',
    'dim_vehicle',
    'fct_fipe_prices',
    'silver_fipe'
)
ORDER BY table_name
""").df()

## 2. Dataset overview


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT vehicle_key) AS distinct_vehicle_keys,
    MIN(data_referencia) AS first_period,
    MAX(data_referencia) AS last_period
FROM vw_fipe_prices_enriched
""").df()

## 3. Monthly record volume


In [ ]:
monthly_volume = con.sql("""
SELECT
    data_referencia,
    rows
FROM vw_monthly_market_summary
ORDER BY data_referencia
""").df()

monthly_volume.tail(12)

## 4. Median vehicle price by month


In [ ]:
monthly_median_price = con.sql("""
SELECT
    data_referencia,
    median_price_brl
FROM vw_monthly_market_summary
ORDER BY data_referencia
""").df()

monthly_median_price.tail(12)

## 5. Vehicle distribution by type


In [ ]:
con.sql("""
SELECT
    tipo_veiculo,
    rows,
    distinct_vehicles,
    pct,
    median_price_brl
FROM vw_vehicle_type_summary
ORDER BY rows DESC
""").df()

## 6. Latest-period brands


In [ ]:
con.sql("""
SELECT
    nome_marca,
    rows,
    distinct_vehicles,
    distinct_fipe_codes,
    median_price_brl
FROM vw_latest_brand_summary
ORDER BY median_price_brl DESC
LIMIT 20
""").df()

## 7. Latest-period fuel mix


In [ ]:
con.sql("""
SELECT
    nome_combustivel,
    sigla_combustivel,
    rows,
    distinct_vehicles,
    pct,
    median_price_brl
FROM vw_latest_fuel_mix
ORDER BY rows DESC
""").df()

## 8. Dimension cardinalities


In [ ]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM dim_date) AS date_rows,
    (SELECT COUNT(*) FROM dim_vehicle) AS vehicle_rows,
    (SELECT COUNT(*) FROM fct_fipe_prices) AS fact_rows
""").df()

## 9. Referential-integrity check


In [ ]:
con.sql("""
SELECT
    SUM(CASE WHEN d.date_key IS NULL THEN 1 ELSE 0 END)
        AS orphan_date_keys,
    SUM(CASE WHEN v.vehicle_key IS NULL THEN 1 ELSE 0 END)
        AS orphan_vehicle_keys
FROM fct_fipe_prices AS f
LEFT JOIN dim_date AS d
    ON f.date_key = d.date_key
LEFT JOIN dim_vehicle AS v
    ON f.vehicle_key = v.vehicle_key
""").df()

## 10. Fact grain check


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS duplicate_fact_keys
FROM (
    SELECT
        date_key,
        vehicle_key
    FROM fct_fipe_prices
    GROUP BY
        date_key,
        vehicle_key
    HAVING COUNT(*) > 1
)
""").df()

In [ ]:
con.close()